# B2 Jupyter Magic Commands — Quick Start

This notebook demonstrates how to use `jupyter-b2` magic commands to interact
with Backblaze B2 Cloud Storage directly from Jupyter notebooks.

## Installation

```bash
pip install 'jupyter-b2[all]'
```

## Prerequisites: B2 CLI Authentication (Recommended)

The easiest way to use `jupyter-b2` is to authenticate once with the **B2 CLI**.
Your credentials are stored locally and reused automatically by the magic commands:

```bash
# Install B2 CLI (if not already installed)
pip install b2

# Authenticate once — credentials are stored in ~/.b2_account_info
b2 account authorize <your-key-id> <your-application-key>
```

After this, every time you run `%load_ext jupyter_b2` in a notebook,
you'll be **automatically authenticated** — no need to enter credentials again!

## 1. Load the Extension

If you've authenticated with the B2 CLI (see above), you'll see a green
"Authenticated" card immediately. No extra steps needed!

In [ ]:
%load_ext jupyter_b2

## 2. Authentication Options

If auto-authentication didn't work (orange box instead of green), you have
several options:

### Option A: B2 CLI (Recommended — Persistent)

Run this **once** in your terminal:
```bash
pip install b2
b2 account authorize <your-key-id> <your-application-key>
```
Then restart your kernel and `%load_ext jupyter_b2` will auto-authenticate.

### Option B: Environment Variables (Good for CI/Docker)

```bash
export B2_APPLICATION_KEY_ID="your-key-id"
export B2_APPLICATION_KEY="your-app-key"
```

### Option C: Explicit Credentials

```python
%b2 auth --key-id <your-key-id> --key <your-app-key>
```

### Option D: Interactive (Prompts for Input)

```python
%b2 auth
```

In [ ]:
# Check authentication status at any time
%b2 status

## 3. Browse Buckets and Files

In [ ]:
# List all buckets
%b2 buckets

In [ ]:
# List files in a bucket
%b2 ls my-bucket/

In [ ]:
# List recursively
%b2 ls my-bucket/datasets/ --recursive

In [ ]:
# Get file metadata
%b2 info my-bucket/datasets/train.csv

## 4. Load Data Directly into DataFrames

The `%b2_load` magic automatically detects file format from the extension
and loads data into the appropriate data structure.

Supported formats: `.csv`, `.tsv`, `.parquet`, `.json`, `.jsonl`, `.feather`, `.orc`, `.xlsx`

In [ ]:
# Load CSV into pandas DataFrame (default)
df = %b2_load my-bucket/datasets/train.csv
df.head()

In [ ]:
# Load Parquet into Polars DataFrame
df_polars = %b2_load my-bucket/datasets/train.parquet --as polars
df_polars.head()

In [ ]:
# Load JSON config as Python dict
config = %b2_load my-bucket/config/settings.json --as json
config

In [ ]:
# Load text file as string
readme = %b2_load my-bucket/docs/README.md --as text
print(readme)

In [ ]:
# Load raw bytes (e.g., model weights)
data = %b2_load my-bucket/models/weights.bin --as bytes
print(f"Downloaded {len(data)} bytes")

## 5. Save Results Back to B2

Use the `%%b2_save` cell magic to save any variable to B2.
Format is auto-detected from the file extension.

In [ ]:
# Create some results
import pandas as pd

results = pd.DataFrame({
    'model': ['random_forest', 'xgboost', 'lightgbm'],
    'accuracy': [0.92, 0.95, 0.94],
    'f1_score': [0.91, 0.94, 0.93],
})

In [ ]:
%%b2_save my-bucket/results/model_comparison.parquet
results

In [ ]:
%%b2_save my-bucket/results/model_comparison.csv
results

In [ ]:
# Save a dict as JSON
%%b2_save my-bucket/config/experiment_config.json
{"learning_rate": 0.001, "epochs": 50, "batch_size": 32}

## 6. Upload and Download Files

In [ ]:
# Upload a local file
%b2 upload ./model_weights.pt my-bucket/models/v3/weights.pt

In [ ]:
# Download a file
%b2 download my-bucket/models/v3/weights.pt ./downloaded_weights.pt

## 7. Generate Pre-Signed URLs

Share files with time-limited URLs — great for team collaboration.

In [ ]:
# Generate a URL valid for 1 hour (3600 seconds)
%b2 presign my-bucket/results/model_comparison.csv --expires 3600

## 8. fsspec Integration (Bonus)

With `jupyter-b2[fsspec]` installed, you can use `b2://` paths directly
in pandas, polars, dask, and any fsspec-compatible library.

**Note:** fsspec requires explicit credentials (it doesn't use B2 CLI stored credentials).

In [ ]:
import pandas as pd

# Read directly with b2:// protocol
df = pd.read_csv(
    "b2://my-bucket/datasets/train.csv",
    storage_options={
        "key_id": "your-key-id",
        "app_key": "your-app-key",
    }
)
df.head()

In [ ]:
# Write back with b2:// protocol
df.to_parquet(
    "b2://my-bucket/datasets/train_processed.parquet",
    storage_options={
        "key_id": "your-key-id",
        "app_key": "your-app-key",
    }
)

## Help

See all available commands:

In [ ]:
%b2 help